# SHFE / COMEX Precious Metals Spread — Dashboard

Single notebook, three layers kept strictly separate. Run top to bottom.

| Section | Owns | Touches Bloomberg |
|---|---|---|
| **1 — Data retrieval** | ticker registry, BQL calls, snapshot object, caching | yes, and only here |
| **2 — Calculation** | units, month matching, premium, carry, sizing | no |
| **3 — Dashboard** | widgets, charts, formatting | no |

Section 2 receives a `MarketSnapshot` from section 1 and nothing else, so every number is
reproducible from a pickled snapshot with no live session. Section 3 does no arithmetic — it
calls the same `headline()` and `carry()` functions you can call directly from a cell.

Each section is split into **definitions** (run once) and **execution** (rerun freely), so you can
edit a definition cell and re-run without re-pulling data.

### The trade

Short spread = **short SHFE / long COMEX / long USDCNH forward**, sized USD-notional equal:
COMEX ounces = SHFE ounces x (1 + premium).

$$\text{Prem}(T) = \frac{F^{SHFE}_{CNY}(T)\cdot k}{F^{FX}(T)\cdot F^{COMEX}_{USD}(T)} - 1$$

The two futures legs have no first-order FX delta on their own; the USDCNH leg is what makes the
P&L replicate the quoted USD-adjusted premium. With that direction the three rolls sum exactly
(in logs) to the slope of the premium term structure — asserted in section 2.

In [ ]:
import pickle
from dataclasses import dataclass, field
from datetime import date
from typing import Dict, List, Optional, Tuple

import numpy as np
import pandas as pd

---
# SECTION 1 — DATA RETRIEVAL

The only part of this notebook that imports `bql`. Nothing here computes a premium, a ratio or a
carry number; it pulls raw fields and hands back DataFrames.

Everything to verify on the terminal lives in `TICKERS` and `FX_TICKERS` below.

**Both legs are addressed by CME month code** — `f'{code_root}{MONTH_CODE}{year} Comdty'`, using
the [CME codes](https://www.cmegroup.com/month-codes.html):

| | Dec 2026 | Jun 2027 |
|---|---|---|
| SHFE gold | `AUAZ6 Comdty` | `AUAM7 Comdty` |
| SHFE silver | `AGAZ6 Comdty` | `AGAM7 Comdty` |
| COMEX gold | `GCZ6 Comdty` | `GCM7 Comdty` |
| COMEX silver | `SIZ6 Comdty` | `SIM7 Comdty` |

Because both sides are addressed by the same month key, a pair is exact by construction — there is
no chain alignment to get wrong. A month an exchange does not list comes back empty and is named.

**`MONTH_SOURCE`** decides which months to request. `GENERIC_CHAIN` (default) discovers them from
the SHFE generic chain (`AU1 … AU15`), which is authoritative on what SHFE actually lists and costs
one lightweight extra pull. `CALENDAR` just walks consecutive months forward and needs no generics
at all.

**Active contract.** `AUA Comdty` / `AGA Comdty` — the exchange's own designation, used to pick the
current active month rather than inferring it from open interest.

### 1.1 — Ticker registry *(definitions)*

In [ ]:
# ===========================================================================
# TICKER REGISTRY
# ===========================================================================
# Both legs are built from CME month codes -- cmegroup.com/month-codes.html
#
#   f'{code_root}{MONTH_CODE}{year} Comdty'
#
#     SHFE gold   Dec 2026 -> AUAZ6 Comdty      COMEX gold   Dec 2026 -> GCZ6 Comdty
#     SHFE silver Dec 2026 -> AGAZ6 Comdty      COMEX silver Dec 2026 -> SIZ6 Comdty
#
# Because both sides are addressed by the same month key, a pair is exact by
# construction. A month an exchange does not list comes back empty and is named,
# rather than silently dropping out of the curve.
#
# gen_root is only used to DISCOVER which months SHFE lists (MONTH_SOURCE below).
# active is the exchange's designated most-active contract; set to None to fall
# back to the open-interest heuristic in section 2.5.

TICKERS: Dict[str, Dict[str, Optional[str]]] = {
    'GOLD': {
        'shfe_code_root': 'AUA',          # AUAZ6 Comdty
        'shfe_gen_root':  'AU',           # AU1 .. AUn, month discovery only
        'shfe_active':    'AUA Comdty',
        'cmx_code_root':  'GC',           # GCZ6 Comdty
        'cmx_gen_root':   'GC',
        'cmx_active':     'GCA Comdty',
    },
    'SILVER': {
        'shfe_code_root': 'AGA',          # AGAZ6 Comdty
        'shfe_gen_root':  'AG',
        'shfe_active':    'AGA Comdty',
        'cmx_code_root':  'SI',           # SIZ6 Comdty
        'cmx_gen_root':   'SI',
        'cmx_active':     'SIA Comdty',
    },
}

# Which delivery months to request.
#   GENERIC_CHAIN : discover from the SHFE generic chain -- authoritative on what
#                   SHFE actually lists, costs one extra lightweight pull.
#   CALENDAR      : every consecutive month for CALENDAR_MONTHS forward. Needs no
#                   generics at all; unlisted months simply come back empty.
MONTH_SOURCE    = 'GENERIC_CHAIN'
CALENDAR_MONTHS = 15
CURVE_DEPTH     = 15            # depth of the SHFE generic discovery pull

FX_QUOTE_MODE = 'OUTRIGHT'      # 'OUTRIGHT' | 'POINTS'

# Where the FX curve's x-axis comes from.
#   SETTLE_DT : pull SETTLE_DT per tenor and measure days from the SPOT DATE.
#               Correct: a 3M forward does not settle 91 days out, and points
#               are quoted against T+2 rather than today.
#   NOMINAL   : the hardcoded days in FX_TICKERS, measured from today. Fallback
#               only -- used automatically if SETTLE_DT does not resolve.
FX_DAYCOUNT_SOURCE = 'SETTLE_DT'
FX_POINT_DIV  = 10000.0         # pips divisor when FX_QUOTE_MODE == 'POINTS'
FX_SPOT_TICKER = 'USDCNH BGN Curncy'

# tenor -> (nominal days, outright ticker, points ticker)
FX_TICKERS: Dict[str, tuple] = {
    '1W':  (  7, 'USDCNH1W BGN Curncy',  'CNH1W BGN Curncy'),
    '1M':  ( 30, 'USDCNH1M BGN Curncy',  'CNH1M BGN Curncy'),
    '2M':  ( 61, 'USDCNH2M BGN Curncy',  'CNH2M BGN Curncy'),
    '3M':  ( 91, 'USDCNH3M BGN Curncy',  'CNH3M BGN Curncy'),
    '6M':  (182, 'USDCNH6M BGN Curncy',  'CNH6M BGN Curncy'),
    '9M':  (273, 'USDCNH9M BGN Curncy',  'CNH9M BGN Curncy'),
    '1Y':  (365, 'USDCNH12M BGN Curncy', 'CNH12M BGN Curncy'),
    '2Y':  (730, 'USDCNH2Y BGN Curncy',  'CNH2Y BGN Curncy'),
}

# CME contract month codes -- cmegroup.com/month-codes.html
MONTH_CODES = {1: 'F', 2: 'G', 3: 'H', 4: 'J', 5: 'K', 6: 'M',
               7: 'N', 8: 'Q', 9: 'U', 10: 'V', 11: 'X', 12: 'Z'}

YEAR_DIGITS = 1                 # AUAZ6 Comdty (1) vs AUAZ26 Comdty (2)


def month_ticker(code_root: str, month_key: str) -> str:
    """'AUA', '2026-12' -> 'AUAZ6 Comdty'.  'SI', '2026-12' -> 'SIZ6 Comdty'."""
    y, m = month_key.split('-')
    return f'{code_root}{MONTH_CODES[int(m)]}{y[-YEAR_DIGITS:]} Comdty'


def month_tickers(code_root: str, month_keys) -> Dict[str, str]:
    """month_key -> ticker, in the order given."""
    return {mk: month_ticker(code_root, mk) for mk in month_keys}


def calendar_months(asof, n: int = CALENDAR_MONTHS) -> List[str]:
    """Consecutive delivery months from the current one, as 'YYYY-MM'."""
    start = pd.Timestamp(asof).to_period('M')
    return [(start + k).strftime('%Y-%m') for k in range(n)]


# Raw fields requested per futures contract. Column names on the left are what
# layer 2 expects; the BQL field is on the right.
FUTURES_FIELDS = {
    'px':       'px_last',
    'contract': 'fut_cur_gen_ticker',
    'month_yr': 'fut_month_yr',
    'expiry':   'last_tradeable_dt',
    'notice':   'fut_notice_first',
    'oi':       'open_int',
    'volume':   'px_volume',
    'ccy':      'crncy',
    'lot':      'fut_cont_size',
    'exch':     'exch_code',
    'name':     'name',
}


def show_registry(n: int = 4) -> pd.DataFrame:
    """Every ticker this notebook will request. Paste into BBG and check."""
    months = calendar_months(date.today(), n)
    rows = []
    for metal, t in TICKERS.items():
        for leg in ('shfe', 'cmx'):
            ex = [month_ticker(t[f'{leg}_code_root'], m) for m in months]
            rows.append(dict(metal=metal, leg=leg, kind='month-coded curve',
                             root=t[f'{leg}_code_root'],
                             tickers=', '.join(ex) + ' ...'))
            if t.get(f'{leg}_active'):
                rows.append(dict(metal=metal, leg=leg, kind='active contract',
                                 root=t[f'{leg}_code_root'],
                                 tickers=t[f'{leg}_active']))
        if MONTH_SOURCE == 'GENERIC_CHAIN':
            rows.append(dict(metal=metal, leg='shfe', kind='month discovery',
                             root=t['shfe_gen_root'],
                             tickers=f"{t['shfe_gen_root']}1 Comdty .. "
                                     f"{t['shfe_gen_root']}{CURVE_DEPTH} Comdty"))
    idx = 1 if FX_QUOTE_MODE == 'OUTRIGHT' else 2
    rows.append(dict(metal='-', leg='fx', kind='fx spot', root='',
                     tickers=FX_SPOT_TICKER))
    rows.append(dict(metal='-', leg='fx', kind=f'fx fwd ({FX_QUOTE_MODE})', root='',
                     tickers=', '.join(v[idx] for v in FX_TICKERS.values())))
    return pd.DataFrame(rows)

### 1.2 — Snapshot container *(definitions)*

The interface between sections 1 and 2.

In [ ]:
# ===========================================================================
# SNAPSHOT CONTAINER  --  the interface between layer 1 and layer 2
# ===========================================================================
@dataclass
class MarketSnapshot:
    """Raw market data. No derived quantities. Layer 2 consumes only this."""
    asof:    pd.Timestamp
    source:  str                                     # 'BQL' | 'MOCK'
    futures: Dict[str, Dict[str, pd.DataFrame]] = field(default_factory=dict)
    fx:      Optional[pd.DataFrame] = None           # tenor, days, outright
    fx_spot: Optional[float] = None
    fx_spot_date: Optional[pd.Timestamp] = None      # T+2; origin of the fx curve
    active:  Dict[str, dict] = field(default_factory=dict)   # metal -> leg -> {...}
    months:  Dict[str, List[str]] = field(default_factory=dict)  # metal -> requested
    errors:  List[str] = field(default_factory=list)

    def raw(self, metal: str, leg: str) -> pd.DataFrame:
        """leg is 'shfe' or 'cmx'. Returns the untouched pull."""
        return self.futures[metal][leg]

    def metals(self) -> List[str]:
        return list(self.futures)

    def active_month(self, metal: str, leg: str = 'shfe') -> Optional[str]:
        """Delivery month 'YYYY-MM' of the exchange's active contract, or None
        if the active ticker was not configured or did not resolve."""
        return (self.active.get(metal, {}).get(leg) or {}).get('month_key')

    def active_table(self) -> pd.DataFrame:
        rows = []
        for metal, legs in self.active.items():
            for leg, d in legs.items():
                rows.append(dict(metal=metal, leg=leg, **(d or {})))
        return pd.DataFrame(rows)

    def summary(self) -> pd.DataFrame:
        rows = []
        for metal, legs in self.futures.items():
            for leg, df in legs.items():
                rows.append(dict(
                    metal=metal, leg=leg, contracts=len(df),
                    ccy=df['ccy'].dropna().unique().tolist() if 'ccy' in df else [],
                    exch=df['exch'].dropna().unique().tolist() if 'exch' in df else [],
                    lot=df['lot'].dropna().unique().tolist() if 'lot' in df else [],
                    first=df['contract'].iloc[0] if len(df) else None,
                    last=df['contract'].iloc[-1] if len(df) else None,
                ))
        return pd.DataFrame(rows)

### 1.3 — Providers *(definitions)*

`MockProvider` implements the same interface as `BQLProvider`, so sections 2 and 3 can be wired up without a Bloomberg session and never know which one produced the data.

In [ ]:
# ===========================================================================
# PROVIDERS
# ===========================================================================
class BQLProvider:
    """Live Bloomberg via BQuant."""
    source = 'BQL'

    def __init__(self, bq=None):
        if bq is None:
            import bql
            bq = bql.Service()
        self.bq = bq
        import bql as _bql
        self._bql = _bql

    # -- low level ---------------------------------------------------------
    def _frame(self, tickers, fields: Dict[str, str]) -> pd.DataFrame:
        """fields: {column_name: bql_field_name}. Assembled BY FIELD NAME, not
        by position, so a reordered response cannot transpose the data."""
        flds = {col: getattr(self.bq.data, fn)() for col, fn in fields.items()}
        res = self.bq.execute(self._bql.Request(tickers, flds))
        df = pd.concat([item.df()[item.name()] for item in res], axis=1)
        df.columns = list(flds)
        return df.reindex(tickers)

    def _frame_tolerant(self, tickers, fields: Dict[str, str]) -> pd.DataFrame:
        """Batch pull that survives an unlisted ticker. Explicit month codes can
        reference contracts an exchange does not list (COMEX silver June, most
        of the year); one bad ticker must not kill the whole request."""
        try:
            return self._frame(tickers, fields)
        except Exception:
            out = []
            for tk in tickers:
                try:
                    out.append(self._frame([tk], fields))
                except Exception:
                    out.append(pd.DataFrame(index=[tk], columns=list(fields)))
            return pd.concat(out)

    def history(self, tickers, fields: Dict[str, str], start, end) -> pd.DataFrame:
        """Date-range form that actually works in BQL. The
        bq.data.px_last(dates=<mixed range>) form returns empty frames."""
        params = {'fill': 'na', 'dates': self.bq.func.range(start, end)}
        flds = {col: getattr(self.bq.data, fn)(**params) for col, fn in fields.items()}
        res = self.bq.execute(self._bql.Request(tickers, flds))
        return pd.concat([item.df() for item in res], axis=1)

    # -- validation --------------------------------------------------------
    def validate(self, n: int = 3) -> pd.DataFrame:
        """Ping the configured tickers: the first `n` month-coded contracts on
        each leg, each active-contract ticker, the SHFE discovery generics, and
        every FX ticker. Check name / ccy / exch / lot before trusting anything."""
        rows = []
        meta = {'name': 'name', 'px': 'px_last', 'ccy': 'crncy',
                'exch': 'exch_code', 'lot': 'fut_cont_size',
                'contract': 'fut_cur_gen_ticker', 'month_yr': 'fut_month_yr'}

        def ping(group, kind, tk, flds=meta):
            try:
                d = self._frame([tk], flds)
                r = dict(group=group, kind=kind, ticker=tk, ok=True)
                r.update({k: d[k].iloc[0] for k in flds})
                rows.append(r)
            except Exception as e:
                rows.append(dict(group=group, kind=kind, ticker=tk, ok=False,
                                 name=str(e)[:70]))

        months = calendar_months(date.today(), n)
        for metal, t in TICKERS.items():
            for leg in ('shfe', 'cmx'):
                for mk in months:
                    ping(metal, f'{leg} month-coded',
                         month_ticker(t[f'{leg}_code_root'], mk))
                if t.get(f'{leg}_active'):
                    ping(metal, f'{leg} active', t[f'{leg}_active'])
            if MONTH_SOURCE == 'GENERIC_CHAIN':
                for i in range(1, n + 1):
                    ping(metal, 'shfe discovery', f"{t['shfe_gen_root']}{i} Comdty")

        idx = 1 if FX_QUOTE_MODE == 'OUTRIGHT' else 2
        for tk in [FX_SPOT_TICKER] + [v[idx] for v in FX_TICKERS.values()]:
            ping('FX', 'fx', tk, {'name': 'name', 'px': 'px_last'})
        return pd.DataFrame(rows)

    # -- pulls -------------------------------------------------------------
    def futures_chain(self, root: str, depth: int = CURVE_DEPTH) -> pd.DataFrame:
        """Generic chain. Used only to discover which months SHFE lists."""
        tickers = [f'{root}{i} Comdty' for i in range(1, depth + 1)]
        return self._frame(tickers, FUTURES_FIELDS)

    def futures_by_month(self, code_root: str, month_keys) -> pd.DataFrame:
        """Explicit month-coded tickers for the months requested. Leaves an
        all-NaN row in place for any month the exchange does not list, so the
        gap is visible and named rather than silently absent."""
        mp = month_tickers(code_root, month_keys)
        df = self._frame_tolerant(list(mp.values()), FUTURES_FIELDS)
        df['requested_month'] = pd.Series({v: k for k, v in mp.items()})
        return df

    def active_contract(self, ticker: str) -> dict:
        """Resolve an active-contract ticker to the contract it points at."""
        d = self._frame([ticker], {'contract': 'fut_cur_gen_ticker',
                                   'month_yr': 'fut_month_yr',
                                   'expiry':   'last_tradeable_dt',
                                   'px':       'px_last',
                                   'oi':       'open_int'})
        exp = pd.to_datetime(d['expiry'].iloc[0])
        s = str(d['month_yr'].iloc[0]).strip().upper()
        mk = pd.to_datetime(s, format='%b %y', errors='coerce')
        if pd.isna(mk):
            mk = pd.to_datetime(s, format='%b %Y', errors='coerce')
        return dict(ticker=ticker, contract=d['contract'].iloc[0],
                    month_key=(mk.strftime('%Y-%m') if pd.notna(mk)
                               else exp.strftime('%Y-%m')),
                    expiry=exp, px=d['px'].iloc[0], oi=d['oi'].iloc[0])

    def fx_curve(self):
        """USDCNH curve anchored on the SPOT DATE, with each tenor placed at its
        actual SETTLE_DT rather than a nominal day count. Returns
        (frame, spot, spot_date)."""
        idx = 1 if FX_QUOTE_MODE == 'OUTRIGHT' else 2
        tks = [FX_SPOT_TICKER] + [v[idx] for v in FX_TICKERS.values()]
        flds = {'px': 'px_last'}
        if FX_DAYCOUNT_SOURCE == 'SETTLE_DT':
            flds['settle'] = 'settle_dt'
        raw = self._frame_tolerant(tks, flds)

        spot = float(raw.loc[FX_SPOT_TICKER, 'px'])
        spot_dt = pd.to_datetime(raw.loc[FX_SPOT_TICKER].get('settle'), errors='coerce')
        if pd.isna(spot_dt):
            spot_dt = pd.Timestamp(date.today())
            print('FX: SETTLE_DT unavailable on spot -- falling back to nominal '
                  'day counts measured from today.')

        rows = [dict(tenor='SP', days=0, days_nominal=0, settle=spot_dt,
                     outright=spot, ticker=FX_SPOT_TICKER)]
        for tenor, (nom, o_tk, p_tk) in FX_TICKERS.items():
            tk = o_tk if idx == 1 else p_tk
            if tk not in raw.index:
                continue
            v = raw.loc[tk, 'px']
            if pd.isna(v):
                continue
            out = float(v) if idx == 1 else spot + float(v) / FX_POINT_DIV
            settle = pd.to_datetime(raw.loc[tk].get('settle'), errors='coerce')
            days = int((settle - spot_dt).days) if pd.notna(settle) else nom
            rows.append(dict(tenor=tenor, days=days, days_nominal=nom,
                             settle=settle, outright=out, ticker=tk))

        fx = pd.DataFrame(rows).sort_values('days').reset_index(drop=True)
        fx['points'] = fx['outright'] - spot
        fx['drift_d'] = fx['days'] - fx['days_nominal']
        return fx, spot, spot_dt


class MockProvider:
    """Synthetic curves with the same interface as BQLProvider, honouring the
    real listing cycles so the unlisted-month path gets exercised. For wiring
    sections 2 and 3 without a Bloomberg session. Never use these numbers."""
    source = 'MOCK'

    _BASE = {'GOLD':   dict(shfe=560.0,  cmx=2450.0, shfe_lot=1000.0, cmx_lot=100.0),
             'SILVER': dict(shfe=8600.0, cmx=30.5,   shfe_lot=15.0,   cmx_lot=5000.0)}

    # SHFE lists consecutive months; COMEX gold Feb/Apr/Jun/Aug/Oct/Dec, COMEX
    # silver Jan/Mar/May/Jul/Sep/Dec, both plus current and next two months.
    _CYCLE = {'GC': {2, 4, 6, 8, 10, 12}, 'SI': {1, 3, 5, 7, 9, 12}}

    def __init__(self, asof=None):
        self.asof = pd.Timestamp(asof or date.today())

    def validate(self, n: int = 3) -> pd.DataFrame:
        return pd.DataFrame([dict(group='MOCK', kind='-', ticker='-', ok=True,
                                  name='MockProvider: no tickers hit')])

    # Roots the mock "exchange" knows about. Hardcoded rather than read from
    # TICKERS, so pointing TICKERS at a bad root produces no data -- which is
    # what a real bad ticker does, and what the data checker needs to see.
    _KNOWN = {'AUA': ('GOLD', 'shfe'), 'AU': ('GOLD', 'shfe'),
              'AGA': ('SILVER', 'shfe'), 'AG': ('SILVER', 'shfe'),
              'GC':  ('GOLD', 'cmx'),   'SI':  ('SILVER', 'cmx'),
              'GCA': ('GOLD', 'cmx'),   'SIA': ('SILVER', 'cmx')}

    def _leg_of(self, code_root):
        return self._KNOWN.get(code_root)

    def _row(self, metal, leg, code_root, mk):
        b = self._BASE[metal]
        base, lot = b[leg], b[f'{leg}_lot']
        slope = 0.022 if leg == 'shfe' else 0.020
        exp = (pd.Timestamp(mk + '-01') + pd.offsets.MonthBegin(0)
               + pd.Timedelta(days=14))
        k = max((exp.to_period('M') - self.asof.to_period('M')).n, 0)
        return dict(px=base * (1 + slope * (k + 1) / 12.0),
                    contract=f'{code_root}{exp.strftime("%b%y").upper()}',
                    month_yr=exp.strftime('%b %y').upper(),
                    expiry=exp, notice=exp - pd.Timedelta(days=2),
                    oi=50000 * (0.7 ** k), volume=20000 * (0.6 ** k),
                    ccy='CNY' if leg == 'shfe' else 'USD',
                    lot=lot, exch='MOCK', name=f'MOCK {metal} {leg.upper()} {mk}')

    def futures_chain(self, root: str, depth: int = CURVE_DEPTH) -> pd.DataFrame:
        known = self._leg_of(root)
        idx = [f'{root}{i+1} Comdty' for i in range(depth)]
        if known is None:
            return pd.DataFrame([{c: np.nan for c in FUTURES_FIELDS} for _ in idx],
                                index=idx)
        metal, leg = known
        months = calendar_months(self.asof, depth)
        return pd.DataFrame([self._row(metal, leg, root, mk) for mk in months], index=idx)

    def futures_by_month(self, code_root: str, month_keys) -> pd.DataFrame:
        known = self._leg_of(code_root)
        if known is None:
            mp = month_tickers(code_root, month_keys)
            rows = []
            for mk, tk in mp.items():
                r = {c: np.nan for c in FUTURES_FIELDS}; r['requested_month'] = mk
                rows.append(r)
            return pd.DataFrame(rows, index=list(mp.values()))
        metal, leg = known
        cycle = self._CYCLE.get(code_root)
        near = {(self.asof + pd.DateOffset(months=k)).strftime('%Y-%m')
                for k in range(3)}
        mp = month_tickers(code_root, month_keys)
        rows, idx = [], []
        for mk, tk in mp.items():
            listed = (cycle is None or int(mk.split('-')[1]) in cycle or mk in near)
            r = self._row(metal, leg, code_root, mk) if listed else \
                {c: np.nan for c in FUTURES_FIELDS}
            r['requested_month'] = mk
            rows.append(r); idx.append(tk)
        return pd.DataFrame(rows, index=idx)

    def active_contract(self, ticker: str) -> dict:
        """Mock: pretend the active contract is the 3rd listed month, so the
        BBG_ACTIVE path differs from the OI front and gets exercised."""
        root = ticker.split()[0]
        known = self._leg_of(root)
        if known is None:
            raise ValueError(f'unknown active ticker {ticker!r}')
        metal, leg = known
        mk = calendar_months(self.asof, 4)[2]
        r = self._row(metal, leg, root, mk)
        return dict(ticker=ticker, contract=r['contract'], month_key=mk,
                    expiry=r['expiry'], px=r['px'], oi=r['oi'])

    _TENOR_OFFSET = {'1W': pd.DateOffset(weeks=1), '1M': pd.DateOffset(months=1),
                     '2M': pd.DateOffset(months=2), '3M': pd.DateOffset(months=3),
                     '6M': pd.DateOffset(months=6), '9M': pd.DateOffset(months=9),
                     '1Y': pd.DateOffset(years=1),  '2Y': pd.DateOffset(years=2)}

    def fx_curve(self):
        """Mock: real calendar offsets from a T+2 spot date, so the settle-date
        path and its drift vs nominal days both get exercised."""
        spot = 7.12
        spot_dt = self.asof + pd.Timedelta(days=2)
        rows = [dict(tenor='SP', days=0, days_nominal=0, settle=spot_dt,
                     outright=spot, ticker='MOCK')]
        for tenor, (nom, _, _) in FX_TICKERS.items():
            settle = spot_dt + self._TENOR_OFFSET[tenor]
            days = int((settle - spot_dt).days)
            rows.append(dict(tenor=tenor, days=days, days_nominal=nom, settle=settle,
                             outright=spot * (1 - 0.012 * days / 365.0), ticker='MOCK'))
        fx = pd.DataFrame(rows).sort_values('days').reset_index(drop=True)
        fx['points'] = fx['outright'] - spot
        fx['drift_d'] = fx['days'] - fx['days_nominal']
        return fx, spot, spot_dt

### 1.4 — Fetch and persist *(definitions)*

In [ ]:
# ===========================================================================
# FETCH / PERSIST
# ===========================================================================
def _raw_month_keys(df: pd.DataFrame) -> pd.Series:
    """Delivery month 'YYYY-MM' straight off a raw pull. Layer 1 needs this to
    know which month-coded tickers to build; layer 2 derives it again
    independently when it normalises."""
    exp = pd.to_datetime(df['expiry'], errors='coerce')
    if 'month_yr' not in df:
        return exp.dt.strftime('%Y-%m')
    s = df['month_yr'].astype(str).str.strip().str.upper()
    mk = pd.to_datetime(s, format='%b %y', errors='coerce')
    mk = mk.fillna(pd.to_datetime(s, format='%b %Y', errors='coerce'))
    return mk.dt.strftime('%Y-%m').fillna(exp.dt.strftime('%Y-%m'))


def discover_months(metal: str, provider, asof, depth: int = CURVE_DEPTH) -> List[str]:
    """Which delivery months to request on both legs."""
    if MONTH_SOURCE == 'CALENDAR':
        return calendar_months(asof, CALENDAR_MONTHS)
    gen = provider.futures_chain(TICKERS[metal]['shfe_gen_root'], depth)
    gen = gen.dropna(subset=['expiry'])
    return list(dict.fromkeys(_raw_month_keys(gen).dropna()))


def fetch(metals=None, provider=None, depth: int = CURVE_DEPTH) -> MarketSnapshot:
    """Discover the delivery months, then pull BOTH legs month-coded against
    the same month keys. Pairing is therefore exact by construction."""
    provider = provider or BQLProvider()
    metals = metals or list(TICKERS)
    snap = MarketSnapshot(asof=pd.Timestamp(date.today()), source=provider.source)

    for metal in metals:
        t = TICKERS[metal]

        try:
            months = discover_months(metal, provider, snap.asof, depth)
        except Exception as e:
            snap.errors.append(f'{metal}/month discovery: {e}')
            months = calendar_months(snap.asof, CALENDAR_MONTHS)
        snap.months[metal] = months

        legs = {}
        for leg in ('shfe', 'cmx'):
            root = t[f'{leg}_code_root']
            try:
                legs[leg] = provider.futures_by_month(root, months)
            except Exception as e:
                snap.errors.append(f'{metal}/{leg} ({root}): {e}')
                legs[leg] = pd.DataFrame()
        snap.futures[metal] = legs

        for leg in ('shfe', 'cmx'):
            df = legs[leg]
            if len(df) and 'requested_month' in df:
                miss = df.loc[df['px'].isna(), 'requested_month'].tolist()
                if miss:
                    root = t[f'{leg}_code_root']
                    print(f'[{metal}/{leg}] not listed: '
                          + ', '.join(f'{m} ({month_ticker(root, m)})' for m in miss))

        act = {}
        for leg in ('shfe', 'cmx'):
            tk = t.get(f'{leg}_active')
            if not tk:
                act[leg] = None
                continue
            try:
                act[leg] = provider.active_contract(tk)
            except Exception as e:
                # has a documented fallback (MAX_OI), so this is a warning
                snap.errors.append(f'[warn] {metal}/{leg} active ({tk}): {e}')
                act[leg] = None
        snap.active[metal] = act

    try:
        snap.fx, snap.fx_spot, snap.fx_spot_date = provider.fx_curve()
    except Exception as e:
        snap.errors.append(f'fx: {e}')

    if snap.errors:
        print('Data layer errors:')
        for e in snap.errors:
            print('  -', e)
    return snap


def save(snap: MarketSnapshot, path: str) -> str:
    """Pickle plain pandas objects only -- never the container class itself, so a
    cached snapshot reloads even if the class is redefined or the kernel differs."""
    payload = dict(asof=snap.asof, source=snap.source, fx=snap.fx,
                   fx_spot=snap.fx_spot, fx_spot_date=snap.fx_spot_date,
                   errors=list(snap.errors),
                   active=snap.active, months=snap.months,
                   futures={m: dict(legs) for m, legs in snap.futures.items()})
    with open(path, 'wb') as f:
        pickle.dump(payload, f)
    return path


def load(path: str) -> MarketSnapshot:
    with open(path, 'rb') as f:
        p = pickle.load(f)
    return MarketSnapshot(asof=p['asof'], source=p['source'], futures=p['futures'],
                          fx=p['fx'], fx_spot=p['fx_spot'],
                          fx_spot_date=p.get('fx_spot_date'),
                          active=p.get('active', {}), months=p.get('months', {}),
                          errors=p.get('errors', []))

### 1.5 — Data checker *(definitions)*

Two stages sharing one report format: `preflight(provider)` pings every configured ticker before a full pull, and `check_data(snap)` audits what came back. Every `FAIL` names the exact config key to change.

In [ ]:
# ===========================================================================
# DATA CHECK
# ===========================================================================
# Two stages, one report format:
#   preflight(provider)  -- before pulling: does every configured ticker resolve?
#   check_data(snap)     -- after pulling:  is what came back usable?
# Every FAIL names the exact config key to change.

_OK, _WARN, _FAIL = 'OK', 'WARN', 'FAIL'
_EXPECTED_CCY = {'shfe': 'CNY', 'cmx': 'USD'}


def _r(area, item, status, detail='', action=''):
    return dict(area=area, item=item, status=status, detail=detail, action=action)


def _remedy(group: str, kind: str) -> str:
    kind = str(kind)
    leg = kind.split()[0] if kind.split() else ''
    if 'month-coded' in kind:
        return (f"TICKERS['{group}']['{leg}_code_root']  "
                f"(or YEAR_DIGITS if the year format is wrong)")
    if 'active' in kind:
        return (f"TICKERS['{group}']['{leg}_active']  "
                f"-- set to None to fall back to the OI rule")
    if 'discovery' in kind:
        return (f"TICKERS['{group}']['shfe_gen_root']  "
                f"-- or set MONTH_SOURCE = 'CALENDAR' to skip discovery")
    return "FX_SPOT_TICKER / FX_TICKERS / FX_QUOTE_MODE"


def preflight(provider, n: int = 2) -> pd.DataFrame:
    """Ping every configured ticker BEFORE pulling a full snapshot."""
    try:
        v = provider.validate(n)
    except Exception as e:
        return pd.DataFrame([_r('preflight', 'validate()', _FAIL, str(e)[:80],
                                'check the BQL session')])
    rows = []
    for _, r in v.iterrows():
        grp, kind, tk = r.get('group', ''), r.get('kind', ''), r.get('ticker', '')
        if bool(r.get('ok')):
            det = ' | '.join(str(r.get(k, '')) for k in ('name', 'ccy', 'exch') if r.get(k) == r.get(k))
            rows.append(_r(grp, tk, _OK, det[:70], ''))
        else:
            rows.append(_r(grp, tk, _FAIL, str(r.get('name', ''))[:70], _remedy(grp, kind)))
    return pd.DataFrame(rows)


def check_data(snap) -> pd.DataFrame:
    """Audit a pulled snapshot. Returns one row per check."""
    rows = []

    for e in snap.errors:
        e = str(e)
        if e.startswith('[warn]'):
            rows.append(_r('fetch', 'active ticker', _WARN, e[6:].strip()[:90],
                           "TICKERS[...]['<leg>_active'] -- set to None to use the "
                           'OI rule instead'))
        else:
            rows.append(_r('fetch', 'exception', _FAIL, e[:90],
                           'see the ticker registry in 1.1'))

    for metal in snap.metals():
        t = TICKERS[metal]
        for leg in ('shfe', 'cmx'):
            df = snap.futures[metal].get(leg)
            root = t[f'{leg}_code_root']
            tag = f'{metal}/{leg}'

            if df is None or not len(df):
                rows.append(_r(tag, 'pull', _FAIL, 'empty frame',
                               f"TICKERS['{metal}']['{leg}_code_root'] = {root!r}"))
                continue

            n_req = len(df)
            n_ok = int(df['px'].notna().sum())
            if n_ok == 0:
                rows.append(_r(tag, 'resolution', _FAIL,
                               f'0 of {n_req} month-coded tickers returned a price',
                               f"TICKERS['{metal}']['{leg}_code_root'] = {root!r} "
                               f"-- example built: {month_ticker(root, snap.months[metal][0])}"))
                continue

            miss = (df.loc[df['px'].isna(), 'requested_month'].tolist()
                    if 'requested_month' in df else [])
            if miss:
                rows.append(_r(tag, 'unlisted months', _WARN,
                               ', '.join(f'{m} ({month_ticker(root, m)})' for m in miss)[:90],
                               'expected if outside the exchange listing cycle'))
            rows.append(_r(tag, 'resolution', _OK, f'{n_ok} of {n_req} months priced'))

            ccy = [c for c in df['ccy'].dropna().unique()]
            exp_ccy = _EXPECTED_CCY[leg]
            if ccy and ccy != [exp_ccy]:
                rows.append(_r(tag, 'currency', _FAIL, f'got {ccy}, expected [{exp_ccy}]',
                               f"TICKERS['{metal}']['{leg}_code_root'] likely points at "
                               f'the wrong exchange'))
            else:
                rows.append(_r(tag, 'currency', _OK, exp_ccy))

            lots = [l for l in pd.to_numeric(df['lot'], errors='coerce').dropna().unique()]
            if len(lots) > 1:
                rows.append(_r(tag, 'contract size', _WARN, f'mixed lot sizes {lots}',
                               'chain may span a contract respecification'))
            elif lots:
                rows.append(_r(tag, 'contract size', _OK, f'{lots[0]:,.0f}'))

            exch = [e for e in df['exch'].dropna().unique()]
            rows.append(_r(tag, 'exchange', _OK, ', '.join(map(str, exch))[:40]))

            exp = pd.to_datetime(df['expiry'], errors='coerce').dropna()
            stale = int((exp < snap.asof).sum())
            if stale:
                rows.append(_r(tag, 'expired contracts', _WARN,
                               f'{stale} contract(s) already past last trade date',
                               'harmless, but they are dropped downstream'))

            oi = pd.to_numeric(df['oi'], errors='coerce').dropna()
            if len(oi) and oi.max() == 0:
                rows.append(_r(tag, 'open interest', _WARN, 'all zero',
                               'OI filters in section 2.1 will drop everything'))

        act = (snap.active.get(metal) or {}).get('shfe')
        if act is None:
            rows.append(_r(f'{metal}/shfe', 'active contract', _WARN, 'not resolved',
                           f"TICKERS['{metal}']['shfe_active'] -- BBG_ACTIVE will "
                           f'fall back to MAX_OI'))
        elif act['month_key'] not in snap.months.get(metal, []):
            rows.append(_r(f'{metal}/shfe', 'active contract', _WARN,
                           f"points at {act['month_key']} ({act['contract']}), "
                           f'outside the requested months',
                           'raise CURVE_DEPTH / CALENDAR_MONTHS'))
        else:
            rows.append(_r(f'{metal}/shfe', 'active contract', _OK,
                           f"{act['contract']} ({act['month_key']})"))

    # -- FX ----------------------------------------------------------------
    fx = snap.fx
    if fx is None or not len(fx):
        rows.append(_r('fx', 'pull', _FAIL, 'no curve', 'FX_SPOT_TICKER / FX_TICKERS'))
    else:
        got = set(fx['tenor']) - {'SP'}
        missing = [t for t in FX_TICKERS if t not in got]
        if missing:
            rows.append(_r('fx', 'tenors', _WARN, f'missing {missing}',
                           'FX_TICKERS -- check the ticker form for those tenors'))
        else:
            rows.append(_r('fx', 'tenors', _OK, f'{len(got)} tenors'))

        if snap.fx_spot_date is None or 'settle' not in fx or fx['settle'].isna().any():
            rows.append(_r('fx', 'settlement dates', _WARN,
                           'SETTLE_DT missing -- using nominal day counts',
                           "FX_DAYCOUNT_SOURCE / the settle_dt field name"))
        else:
            rows.append(_r('fx', 'settlement dates', _OK,
                           f'spot date {pd.Timestamp(snap.fx_spot_date).date()}, '
                           f"max drift {int(fx['drift_d'].abs().max())}d"))

        need = pd.concat([pd.to_datetime(snap.futures[m]['shfe']['expiry'],
                                         errors='coerce')
                          for m in snap.metals()]).dropna()
        if len(need) and snap.fx_spot_date is not None:
            far = int((need.max() - pd.Timestamp(snap.fx_spot_date)).days)
            cover = int(fx['days'].max())
            if far > cover:
                rows.append(_r('fx', 'curve coverage', _WARN,
                               f'longest expiry {far}d vs curve {cover}d '
                               f'-- flat extrapolation beyond',
                               'extend FX_TICKERS, or ignore if you trade inside 1Y'))
            else:
                rows.append(_r('fx', 'curve coverage', _OK, f'{cover}d covers {far}d'))

        if float(fx.loc[fx['tenor'] == 'SP', 'outright'].iloc[0]) <= 0:
            rows.append(_r('fx', 'spot', _FAIL, 'non-positive', 'FX_SPOT_TICKER'))

    return pd.DataFrame(rows)


def print_check(report: pd.DataFrame, show_ok: bool = False) -> None:
    n = report['status'].value_counts().to_dict()
    print(f"{n.get(_OK,0)} OK   {n.get(_WARN,0)} WARN   {n.get(_FAIL,0)} FAIL\n")
    for status in (_FAIL, _WARN, _OK):
        if status == _OK and not show_ok:
            continue
        sub = report[report['status'] == status]
        if not len(sub):
            continue
        print(f'--- {status} ---')
        for _, r in sub.iterrows():
            print(f"  [{r['area']}] {r['item']}: {r['detail']}")
            if r['action']:
                print(f"      -> change {r['action']}")
        print()


def require_clean(report: pd.DataFrame, strict: bool = True) -> bool:
    """Call before section 2. Raises on FAIL so nothing downstream runs on bad data."""
    bad = report[report['status'] == _FAIL]
    if len(bad) and strict:
        raise AssertionError(
            f'{len(bad)} data check failure(s) -- fix the config keys listed above '
            f'and re-run the pull. First: {bad.iloc[0]["item"]} / {bad.iloc[0]["detail"]}')
    return not len(bad)

### 1.6 — Run it

Every ticker this notebook will request, before anything is pulled.

In [ ]:
show_registry()

Ping the configured tickers: the first few generics of each chain, each active-contract ticker,
and every FX ticker. Check `name`, `ccy`, `exch` and `lot` look right, and that the `active` rows
resolve to a sensible `contract` and `month_yr`. The registry cell above is the only place tickers
are defined.

In [ ]:
provider = BQLProvider()        # swap to MockProvider() to wire up offline

pre = preflight(provider)
print_check(pre, show_ok=True)

Pull. Raw fields only, nothing derived.

In [ ]:
snap = fetch(['SILVER', 'GOLD'], provider=provider)
snap.summary()

Audit what actually came back. `require_clean` raises on any `FAIL`, so nothing downstream runs on bad data — comment it out if you want to inspect a broken pull.

In [ ]:
report = check_data(snap)
print_check(report)
require_clean(report)

In [ ]:
report      # full table: area, item, status, detail, action

Eyeball the raw chains against `CT` / `DES` before trusting anything downstream.

In [ ]:
snap.raw('SILVER', 'shfe')[['name','contract','month_yr','expiry','notice',
                            'px','oi','volume','ccy','lot','exch']]

In [ ]:
snap.raw('SILVER', 'cmx')[['name','contract','month_yr','expiry','px','oi','ccy','lot']]

In [ ]:
print(f"FX spot date (curve origin): {snap.fx_spot_date}")
snap.fx[['tenor','ticker','settle','days','days_nominal','drift_d','outright','points']]

The FX curve is anchored on the **spot date**, and each tenor sits at its actual `SETTLE_DT`
rather than a nominal day count. `drift_d` is the gap between the two — that is how far each point
would have been mis-plotted on the old nominal axis.

Cache it, so sections 2 and 3 are reproducible without a live session.

In [ ]:
save(snap, 'snap_latest.pkl')
# snap = load('snap_latest.pkl')

---
# SECTION 2 — CALCULATION METHODOLOGY

Pure functions over the snapshot. No BQL calls, no widgets. Nothing below reaches back into
section 1 except through the `snap` object.

### 2.1 — Contract specs and unit conversion *(definitions)*

Quote conventions and sizing, not tickers. Lot sizes are cross-checked against `FUT_CONT_SIZE` by `verify_specs()`.

In [ ]:
G_PER_OZ = 31.1034768

# ---------------------------------------------------------------------------
# CONTRACT SPECS  --  quote conventions and sizing, not tickers.
# Lot sizes are cross-checked against FUT_CONT_SIZE by verify_specs().
# ---------------------------------------------------------------------------
SPECS: Dict[str, dict] = {
    'GOLD': dict(
        shfe_lot=1000.0,        # grams per lot
        shfe_px_per='GRAM',     # quoted CNY per gram
        cmx_lot=100.0,          # troy oz per lot
        ceiling_pct=None,       # no clean import arb: PBoC quota-restricted
    ),
    'SILVER': dict(
        shfe_lot=15.0,          # kg per lot
        shfe_px_per='KG',       # quoted CNY per kg
        cmx_lot=5000.0,
        ceiling_pct=13.0,       # import + SHFE delivery arb ceiling
    ),
}

MIN_OI_SHFE  = 1000     # "active contract" floor on the SHFE chain
MIN_OI_CMX   = 0        # COMEX lists every month -- do not filter
ACTIVE_RULE  = 'BBG_ACTIVE'   # 'BBG_ACTIVE' | 'MAX_OI' | 'FRONT'
#   BBG_ACTIVE : the month the exchange's active ticker (AUA / AGA) points at.
#                Authoritative; falls back to MAX_OI if unavailable.
#   MAX_OI     : highest open interest among contracts clearing the OI floor.
#   FRONT      : nearest expiry.


# ===========================================================================
# UNIT CONVERSION
# ===========================================================================
def shfe_cny_per_oz(px: float, px_per: str) -> float:
    if px_per == 'GRAM':
        return px * G_PER_OZ
    if px_per == 'KG':
        return px * G_PER_OZ / 1000.0
    raise ValueError(f'unknown quote basis {px_per!r}')


def shfe_oz_per_lot(metal: str) -> float:
    s = SPECS[metal]
    if s['shfe_px_per'] == 'GRAM':
        return s['shfe_lot'] / G_PER_OZ           # gold  32.15074 oz
    return s['shfe_lot'] * 1000.0 / G_PER_OZ      # silver 482.2612 oz


def verify_specs(snap, metal: str) -> pd.DataFrame:
    """Cross-check the lot sizes assumed here against FUT_CONT_SIZE from layer 1.
    A mismatch means either the spec or the ticker root is wrong."""
    rows = []
    for leg, assumed in (('shfe', SPECS[metal]['shfe_lot']),
                         ('cmx',  SPECS[metal]['cmx_lot'])):
        df = snap.raw(metal, leg)
        got = pd.to_numeric(df.get('lot'), errors='coerce').dropna().unique()
        rows.append(dict(metal=metal, leg=leg, assumed=assumed,
                         from_bbg=got.tolist(),
                         match=bool(len(got)) and np.allclose(got, assumed)))
    return pd.DataFrame(rows)

### 2.2 — Normalisation and FX interpolation *(definitions)*

In [ ]:
# ===========================================================================
# NORMALISATION
# ===========================================================================
def _month_key(df: pd.DataFrame) -> pd.Series:
    """Delivery month as 'YYYY-MM'. Prefers FUT_MONTH_YR; falls back to the
    expiry month, valid for SHFE and COMEX metals where the last trading day
    falls inside the delivery month."""
    fallback = df['expiry'].dt.strftime('%Y-%m')
    if 'month_yr' not in df.columns:
        return fallback
    s = df['month_yr'].astype(str).str.strip().str.upper()
    p = pd.to_datetime(s, format='%b %y', errors='coerce')
    p = p.fillna(pd.to_datetime(s, format='%b %Y', errors='coerce'))
    return p.dt.strftime('%Y-%m').fillna(fallback)


def normalise(raw: pd.DataFrame, asof: pd.Timestamp, min_oi: float) -> pd.DataFrame:
    """Raw chain -> typed, keyed, sorted. Adds month_key, dte, active."""
    df = raw.copy()
    df['expiry'] = pd.to_datetime(df['expiry'])
    for c in ('px', 'oi', 'volume', 'lot'):
        if c in df:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['px', 'expiry']).copy()
    df['dte']       = (df['expiry'] - asof).dt.days
    df['month_key'] = _month_key(df)
    df['active']    = df['oi'].fillna(0) >= min_oi
    return df.sort_values('expiry')


# ===========================================================================
# FX CURVE
# ===========================================================================
def fx_forward(fx: pd.DataFrame, spot: float, target, origin) -> float:
    """Linear interpolation on forward POINTS against days, so the spot level
    does not leak into the interpolation. Flat extrapolation past the last tenor.

    `origin` must be the FX SPOT DATE (T+2), not today: forward points are
    quoted against spot, and each tenor is plotted at its actual SETTLE_DT.
    Passing today instead shifts every interpolation by about two days.
    """
    d = (pd.Timestamp(target) - pd.Timestamp(origin)).days
    return spot + float(np.interp(d, fx['days'].values,
                                  (fx['outright'] - spot).values))


def fx_origin(snap):
    """Spot date if the data layer got one, else today with the nominal fallback."""
    return snap.fx_spot_date if snap.fx_spot_date is not None else snap.asof

### 2.3 — Delivery-month matching *(definitions)*

Legs pair on delivery month. No proximity fallback: a month either matches or it is reported unmatched, because a Jun-vs-Aug pairing is a different trade rather than an approximation of the right one.

In [ ]:
# ===========================================================================
# MATCHING
# ===========================================================================
def match_by_month(shfe: pd.DataFrame, cmx: pd.DataFrame):
    """Pair on DELIVERY MONTH. Since layer 1 builds the COMEX ticker from the
    SHFE month code, a match here is exact by construction -- this is really a
    join that also reports which requested months the exchange never listed.
    No proximity fallback: a Jun-vs-Jul pairing is a different trade."""
    c = cmx.drop_duplicates(subset='month_key', keep='first').set_index('month_key', drop=False)
    pairs, missing = [], []
    for tk, s in shfe.iterrows():
        k = s['month_key']
        if k in c.index:
            pairs.append((tk, s, c.loc[k]))
        else:
            missing.append(dict(shfe_ticker=tk, contract=s.get('contract'), month=k))
    return pairs, missing


def chain_coverage(snap, metal: str) -> pd.DataFrame:
    """Month-by-month map of what exists on each side.

    Both legs are addressed by explicit month code, so an unlisted month shows
    up as a named ticker that returned nothing -- e.g. SIM6 for COMEX silver
    June, which sits outside the Jan/Mar/May/Sep and Jul/Dec cycle for most of
    the year, while SHFE lists AGAM6 perfectly happily.
    """
    t = TICKERS[metal]
    s = normalise(snap.raw(metal, 'shfe'), snap.asof, MIN_OI_SHFE)
    c = normalise(snap.raw(metal, 'cmx'),  snap.asof, MIN_OI_CMX)
    s_listed, c_listed = set(s['month_key']), set(c['month_key'])
    s_oi = dict(zip(s['month_key'], s['oi']))
    s_act = dict(zip(s['month_key'], s['active']))
    s_con = dict(zip(s['month_key'], s.get('contract', pd.Series(dtype=object))))

    rows = []
    for mk in snap.months.get(metal, sorted(s_listed | c_listed)):
        active = bool(s_act.get(mk, False))
        rows.append(dict(
            month=mk,
            shfe_ticker=month_ticker(t['shfe_code_root'], mk),
            shfe_listed=mk in s_listed,
            shfe_contract=s_con.get(mk),
            shfe_oi=s_oi.get(mk),
            shfe_active=active,
            cmx_ticker=month_ticker(t['cmx_code_root'], mk),
            cmx_listed=mk in c_listed,
            tradeable_pair=active and mk in c_listed,
        ))
    out = pd.DataFrame(rows)
    out.attrs['metal'] = metal
    out.attrs['blocked'] = out.loc[out['shfe_active'] & ~out['cmx_listed'],
                                   ['month', 'shfe_ticker', 'cmx_ticker']].to_dict('records')
    return out

### 2.4 — Premium term structure *(definitions)*

In [ ]:
# ===========================================================================
# PREMIUM TERM STRUCTURE
# ===========================================================================
def premium_curve(snap, metal: str) -> pd.DataFrame:
    """One row per matched delivery month. This is the core table."""
    spec = SPECS[metal]
    shfe = normalise(snap.raw(metal, 'shfe'), snap.asof, MIN_OI_SHFE)
    cmx  = normalise(snap.raw(metal, 'cmx'),  snap.asof, MIN_OI_CMX)
    pairs, missing = match_by_month(shfe[shfe['active']], cmx)

    rows = []
    for tk, s, c in pairs:
        f     = fx_forward(snap.fx, snap.fx_spot, s['expiry'], fx_origin(snap))
        s_usd = shfe_cny_per_oz(s['px'], spec['shfe_px_per']) / f
        rows.append(dict(
            month=s['month_key'],
            shfe_ticker=tk, shfe_contract=s.get('contract'),
            shfe_expiry=s['expiry'], shfe_dte=s['dte'],
            shfe_px_cny=s['px'], shfe_oi=s['oi'],
            fx_fwd=f, shfe_usd_oz=s_usd,
            cmx_contract=c.get('contract'), cmx_expiry=c['expiry'],
            cmx_px_usd=c['px'], cmx_oi=c['oi'],
            expiry_gap_d=int((c['expiry'] - s['expiry']).days),
            prem_usd_oz=s_usd - c['px'],
            prem_pct=(s_usd / c['px'] - 1.0) * 100.0,
        ))
    out = pd.DataFrame(rows)
    if spec['ceiling_pct'] is not None and len(out):
        out['dist_to_ceiling_pct'] = spec['ceiling_pct'] - out['prem_pct']
    out.attrs['unmatched'] = missing
    out.attrs['metal'] = metal
    out.attrs['active_month'] = snap.active_month(metal, 'shfe')
    out.attrs['active_contract'] = (snap.active.get(metal, {}).get('shfe') or {}).get('contract')
    return out

### 2.5 — Current active contract *(definitions)*

`ACTIVE_RULE` defaults to `BBG_ACTIVE` — the delivery month `AUA` / `AGA` currently points at,
which is the exchange's own designation rather than a heuristic.

It falls back to `MAX_OI` if the active ticker is unavailable or lands on a month filtered out of
the curve, and `headline()` reports which rule actually decided via the *Selected by* row. The
`MAX_OI` fallback matters because SHFE gold and silver concentrate open interest in the deferred
Jun/Dec contracts, not the front.

In [ ]:
# ===========================================================================
# CURRENT ACTIVE CONTRACT
# ===========================================================================
def active_idx(pc: pd.DataFrame, rule: Optional[str] = None) -> int:
    """Row index of SHFE's current active contract.

    BBG_ACTIVE uses the delivery month that AUA / AGA currently points at, which
    is the exchange's own designation rather than a heuristic. Falls back to
    MAX_OI when the active ticker is not configured, did not resolve, or lands on
    a month that was filtered out of the curve.
    """
    rule = rule or ACTIVE_RULE
    if rule == 'BBG_ACTIVE':
        m = pc.attrs.get('active_month')
        if m is not None:
            hit = np.flatnonzero((pc['month'] == m).values)
            if len(hit):
                return int(hit[0])
        rule = 'MAX_OI'          # fall back
    if rule == 'MAX_OI':
        return int(np.nan_to_num(pc['shfe_oi'].values.astype(float)).argmax())
    return 0


def active_source(pc: pd.DataFrame, rule: Optional[str] = None) -> str:
    """Which rule actually decided the active contract, after any fallback."""
    rule = rule or ACTIVE_RULE
    if rule == 'BBG_ACTIVE':
        m = pc.attrs.get('active_month')
        if m is not None and (pc['month'] == m).any():
            return f"BBG_ACTIVE ({pc.attrs.get('active_contract')})"
        return 'MAX_OI (fallback: active ticker unavailable or filtered out)'
    return rule


def headline(pc: pd.DataFrame, metal: str, i: Optional[int] = None):
    """Current premium/discount in USD/oz and as % of COMEX, plus its build-up."""
    i = active_idx(pc) if i is None else i
    r, spec = pc.iloc[i], SPECS[metal]
    unit = 'g' if spec['shfe_px_per'] == 'GRAM' else 'kg'
    rows = [
        ('Delivery month',      str(r['month'])),
        ('Selected by',         active_source(pc)),
        ('SHFE contract',       f"{r['shfe_contract']}    OI {r['shfe_oi']:,.0f}"),
        ('COMEX contract',      f"{r['cmx_contract']}    OI {r['cmx_oi']:,.0f}"),
        ('Expiry stub (days)',  f"{r['expiry_gap_d']:+d}"),
        ('', ''),
        ('SHFE price',          f"{r['shfe_px_cny']:,.2f} CNY/{unit}"),
        ('USDCNH fwd @ expiry', f"{r['fx_fwd']:.4f}"),
        ('SHFE in USD/oz',      f"{r['shfe_usd_oz']:,.3f}"),
        ('COMEX in USD/oz',     f"{r['cmx_px_usd']:,.3f}"),
        (' ', ''),
        ('PREMIUM  USD/oz',     f"{r['prem_usd_oz']:+,.3f}"),
        ('PREMIUM  % of COMEX', f"{r['prem_pct']:+.3f} %"),
    ]
    if spec['ceiling_pct'] is not None:
        rows.append(('Distance to ceiling', f"{spec['ceiling_pct'] - r['prem_pct']:+.3f} %"))
    return pd.DataFrame(rows, columns=['', 'Value']).set_index(''), i

### 2.6 — Carry *(definitions)*

Three rolls, in % and in $. Logs, so the legs compose exactly and reconcile against the premium slope.

In [ ]:
# ===========================================================================
# CARRY  --  three rolls, in % and in $
# ===========================================================================
def carry(pc: pd.DataFrame, metal: str, front: Optional[int] = None,
          back: Optional[int] = None, direction: str = 'SHORT',
          shfe_lots: float = 1.0):
    """Roll the whole package from one matched pair to another.

    Short spread:   +dlnF_shfe  (short)   -dlnF_comex (long)   -dlnF_fx (long USDCNH)
    Logs, so the three legs compose exactly and reconcile to the premium slope.
    """
    front = active_idx(pc) if front is None else front
    back  = min(front + 1, len(pc) - 1) if back is None else back
    a, b  = pc.iloc[front], pc.iloc[back]
    sgn   = -1 if direction.upper() == 'SHORT' else +1
    days  = max((pd.Timestamp(b['shfe_expiry']) - pd.Timestamp(a['shfe_expiry'])).days, 1)
    ann   = 365.0 / days

    d_shfe = np.log(b['shfe_px_cny'] / a['shfe_px_cny'])
    d_cmx  = np.log(b['cmx_px_usd']  / a['cmx_px_usd'])
    d_fx   = np.log(b['fx_fwd']      / a['fx_fwd'])

    legs = [('SHFE roll',   'short' if sgn < 0 else 'long',  -sgn * d_shfe),
            ('COMEX roll',  'long'  if sgn < 0 else 'short', +sgn * d_cmx),
            ('USDCNH roll', 'long'  if sgn < 0 else 'short', +sgn * d_fx)]
    net = sum(v for _, _, v in legs)
    legs.append(('NET CARRY', '', net))

    oz       = shfe_lots * shfe_oz_per_lot(metal)
    notional = oz * a['shfe_usd_oz']

    tbl = pd.DataFrame(legs, columns=['Leg', 'Side', '_v']).set_index('Leg')
    tbl['Period %']     = tbl['_v'] * 100
    tbl['Annualised %'] = tbl['_v'] * ann * 100
    tbl['Period $']     = tbl['_v'] * notional
    tbl['Annualised $'] = tbl['_v'] * ann * notional
    tbl['Period $/oz']  = tbl['_v'] * a['shfe_usd_oz']
    tbl['Period $/lot'] = tbl['Period $'] / max(shfe_lots, 1e-12)
    tbl = tbl.drop(columns='_v')

    d_prem = np.log((1 + b['prem_pct'] / 100) / (1 + a['prem_pct'] / 100))
    meta = dict(metal=metal, direction=direction.upper(),
                front_i=front, back_i=back,
                front=f"{a['shfe_contract']} / {a['cmx_contract']}",
                back=f"{b['shfe_contract']} / {b['cmx_contract']}",
                roll_days=days, prem_front=a['prem_pct'], prem_back=b['prem_pct'],
                lots=shfe_lots, oz=oz, notional_usd=notional,
                reconciliation_error=net - (-sgn * d_prem))
    return tbl, meta


def check_reconciliation(pc: pd.DataFrame, metal: str, tol: float = 1e-10) -> float:
    """The three rolls must equal the premium-curve slope. A flipped sign, a bad
    unit conversion or an inverted FX quote all break this instead of quietly
    producing plausible numbers."""
    _, meta = carry(pc, metal)
    err = meta['reconciliation_error']
    if abs(err) > tol:
        raise AssertionError(f'carry legs do not reconcile to premium slope: {err:.3e}')
    return err

### 2.7 — Sizing *(definitions)*

In [ ]:
# ===========================================================================
# SIZING
# ===========================================================================
def size_trade(pc: pd.DataFrame, metal: str, shfe_lots: float,
               i: Optional[int] = None, direction: str = 'SHORT') -> pd.DataFrame:
    """USD-notional equal: COMEX ounces = SHFE ounces * (1 + premium).
    Lot rounding and the residual mismatch are shown, not absorbed."""
    i = active_idx(pc) if i is None else i
    spec, r = SPECS[metal], pc.iloc[i]
    sgn = -1 if direction.upper() == 'SHORT' else +1
    p   = r['prem_pct'] / 100.0

    s_oz  = shfe_lots * shfe_oz_per_lot(metal)
    s_cny = shfe_lots * spec['shfe_lot'] * r['shfe_px_cny']
    s_usd = s_oz * r['shfe_usd_oz']

    c_oz_req  = s_oz * (1.0 + p)
    c_lots_th = c_oz_req / spec['cmx_lot']
    c_lots    = float(np.round(c_lots_th))
    c_oz      = c_lots * spec['cmx_lot']
    c_usd     = c_oz * r['cmx_px_usd']

    return pd.DataFrame([
        ('Direction',             direction.upper()),
        ('Pair',                  f"{r['shfe_contract']}  vs  {r['cmx_contract']}"),
        ('Premium (%)',           f"{r['prem_pct']:+.3f}"),
        ('', ''),
        ('SHFE lots',             f"{sgn*shfe_lots:+,.0f}"),
        ('SHFE ounces',           f"{sgn*s_oz:+,.2f}"),
        ('SHFE notional (CNY)',   f"{sgn*s_cny:+,.0f}"),
        ('SHFE notional (USD)',   f"{sgn*s_usd:+,.0f}"),
        (' ', ''),
        ('COMEX ounces required', f"{-sgn*c_oz_req:+,.2f}"),
        ('COMEX lots (exact)',    f"{-sgn*c_lots_th:+,.4f}"),
        ('COMEX lots (rounded)',  f"{-sgn*c_lots:+,.0f}"),
        ('COMEX notional (USD)',  f"{-sgn*c_usd:+,.0f}"),
        ('Notional mismatch USD', f"{c_usd - s_usd:+,.0f}"),
        ('Notional mismatch %',   f"{(c_usd/s_usd - 1)*100:+.3f}"),
        ('  ', ''),
        ('FX leg',                'BUY USDCNH fwd' if sgn < 0 else 'SELL USDCNH fwd'),
        ('FX forward rate',       f"{r['fx_fwd']:.4f}"),
        ('FX notional (USD)',     f"{-sgn*s_usd:+,.0f}"),
        ('FX notional (CNH)',     f"{sgn*s_usd*r['fx_fwd']:+,.0f}"),
        ('FX value date',         pd.Timestamp(r['shfe_expiry']).strftime('%Y-%m-%d')),
    ], columns=['', 'Value']).set_index('')


def build(snap, metals: Optional[List[str]] = None) -> Dict[str, pd.DataFrame]:
    """Everything layer 3 needs: one premium curve per metal."""
    return {m: premium_curve(snap, m) for m in (metals or snap.metals())}

### 2.8 — Checks

**Check 1 — contract specs.** Compares the lot sizes the maths assumes against `FUT_CONT_SIZE`
from section 1. A mismatch means either the spec or the ticker root is wrong, which is exactly
what a ticker typo looks like.

In [ ]:
pd.concat([verify_specs(snap, m) for m in snap.metals()], ignore_index=True)

**Check 2 — chain coverage.** One row per SHFE delivery month, showing the COMEX ticker built for
it and whether that contract exists. This is where the listing-cycle mismatch shows up: COMEX gold
lists Feb/Apr/Aug/Oct within 23 months and Jun/Dec within 72; COMEX silver lists Jan/Mar/May/Sep
within 23 months and Jul/Dec within 60 — plus, for both, the current and next two calendar months.

The practical consequence for silver: SHFE's liquid June contract has no COMEX counterpart for most
of the year, because June falls outside the SI cycle. `tradeable_pair` is the column that matters.

In [ ]:
for m in snap.metals():
    cov = chain_coverage(snap, m)
    print(f"--- {m} ---")
    display(cov)
    blocked = cov.attrs['blocked']
    if blocked:
        print("  Active SHFE months with no COMEX contract:")
        for b in blocked:
            print(f"    {b['month']}   {b['shfe_ticker']:>16}  ->  {b['cmx_ticker']} (not listed)")
    else:
        print("  Every active SHFE month has a listed COMEX contract.")

### 2.9 — Build the premium curves

In [ ]:
curves = build(snap)
pc = curves['SILVER']
pc[['month','shfe_contract','cmx_contract','expiry_gap_d','shfe_oi','cmx_oi',
    'shfe_px_cny','fx_fwd','shfe_usd_oz','cmx_px_usd','prem_usd_oz','prem_pct']].round(4)

**Check 3 — carry reconciliation.** The three rolls must sum in logs to the slope of the
premium term structure:

$$+\Delta \ln F^{SHFE} \;-\; \Delta \ln F^{COMEX} \;-\; \Delta \ln F^{FX} \;=\; \Delta \ln(1+\text{Prem})$$

Two independent paths to the same number. A flipped leg sign, a bad unit conversion or an inverted
FX quote breaks this rather than quietly producing a plausible figure.

In [ ]:
for m in snap.metals():
    print(f"{m:7s} reconciliation error: {check_reconciliation(curves[m], m):.3e}")

### 2.10 — The two headline outputs, outside the dashboard

Same functions section 3 calls, so these drop into a report or a scheduled job without dragging
`ipywidgets` along.

In [ ]:
hd, i = headline(pc, 'SILVER')
display(hd)

tbl, meta = carry(pc, 'SILVER', i, None, direction='SHORT', shfe_lots=100)
print(f"\n{meta['front']}  ->  {meta['back']}   ({meta['roll_days']} days)")
print(f"Notional per leg: {meta['notional_usd']:,.0f} USD\n")
display(tbl[['Side','Period %','Annualised %','Period $','Annualised $','Period $/lot']].round(3))

---
# SECTION 3 — DASHBOARD

Presentation only. Consumes the premium curves from section 2 and renders them. No arithmetic
happens here — every figure on screen comes from `headline()`, `carry()` and `size_trade()`.

### 3.1 — Dashboard *(definitions)*

In [ ]:
_BANNER = ('<div style="padding:14px 18px;border-radius:8px;background:{bg};'
           'color:#fff;font-family:sans-serif;display:inline-block;margin-right:14px">'
           '<div style="font-size:11px;letter-spacing:1.5px;opacity:.85">{lab}</div>'
           '<div style="font-size:30px;font-weight:600;line-height:1.25">{val}</div>'
           '<div style="font-size:11px;opacity:.85">{sub}</div></div>')


class Dashboard:
    def __init__(self, curves: Dict[str, pd.DataFrame], snap):
        import ipywidgets as w
        self.w, self.curves, self.snap = w, curves, snap

        metals = list(curves)
        self.metal = w.Dropdown(options=metals, value=metals[0], description='Metal:')
        self.dirn  = w.Dropdown(options=['SHORT', 'LONG'], value='SHORT', description='Direction:')
        self.rule  = w.Dropdown(options=['BBG_ACTIVE', 'MAX_OI', 'FRONT'], value=ACTIVE_RULE,
                                description='Active:')
        self.lots  = w.BoundedFloatText(value=100, min=1, max=100000, step=1,
                                        description='SHFE lots:')
        self.roll  = w.Dropdown(description='Roll into:')

        self.out_head  = w.Output()
        self.out_carry = w.Output()
        self.out_size  = w.Output()
        self.out_chart = w.Output()

        for c in (self.dirn, self.roll):
            c.observe(self._redraw, 'value')
        self.lots.observe(self._redraw, 'value')
        for c in (self.metal, self.rule):
            c.observe(self._rebuild_rolls, 'value')

        self.box = w.VBox([
            w.HBox([self.metal, self.dirn, self.rule, self.lots]),
            self.out_head,
            w.HBox([
                w.VBox([w.HTML('<b>Cost of carry — three curves combined</b>'),
                        self.roll, self.out_carry]),
                w.VBox([w.HTML('<b>Trade sizer — USD notional equal</b>'),
                        self.out_size]),
            ]),
            self.out_chart,
        ])
        self._rebuild_rolls()

    # -- charts ------------------------------------------------------------
    def _charts(self, metal):
        import bqplot as bqp
        w = self.w
        pc, spec = self.curves[metal], SPECS[metal]

        xs, ys = bqp.DateScale(), bqp.LinearScale()
        f1 = bqp.Figure(
            title=f'{metal} — outright curves (USD/oz)', legend_location='top-left',
            marks=[bqp.Lines(x=pc['shfe_expiry'].values, y=pc['shfe_usd_oz'].values,
                             scales={'x': xs, 'y': ys}, labels=['SHFE'],
                             display_legend=True, marker='circle'),
                   bqp.Lines(x=pc['cmx_expiry'].values, y=pc['cmx_px_usd'].values,
                             scales={'x': xs, 'y': ys}, labels=['COMEX'],
                             display_legend=True, marker='circle', colors=['#d62728'])],
            axes=[bqp.Axis(scale=xs),
                  bqp.Axis(scale=ys, orientation='vertical', tick_format='.2f')],
            fig_margin={'top': 50, 'bottom': 50, 'left': 70, 'right': 20})

        xs2, ys2 = bqp.DateScale(), bqp.LinearScale()
        marks = [bqp.Lines(x=pc['shfe_expiry'].values, y=pc['prem_pct'].values,
                           scales={'x': xs2, 'y': ys2}, labels=['Premium % of COMEX'],
                           display_legend=True, marker='circle', colors=['#2ca02c'])]
        if spec['ceiling_pct'] is not None:
            marks.append(bqp.Lines(x=pc['shfe_expiry'].values,
                                   y=np.full(len(pc), spec['ceiling_pct']),
                                   scales={'x': xs2, 'y': ys2}, colors=['#999999'],
                                   line_style='dashed', display_legend=True,
                                   labels=[f"{spec['ceiling_pct']:.0f}% ceiling"]))
        f2 = bqp.Figure(
            marks=marks, title=f'{metal} — premium term structure (slope = carry)',
            legend_location='top-left',
            axes=[bqp.Axis(scale=xs2),
                  bqp.Axis(scale=ys2, orientation='vertical', tick_format='.2f', label='%')],
            fig_margin={'top': 50, 'bottom': 50, 'left': 70, 'right': 20})

        fx = self.snap.fx
        xs3, ys3 = bqp.LinearScale(), bqp.LinearScale()
        f3 = bqp.Figure(
            title='USDCNH forward curve', legend_location='top-left',
            marks=[bqp.Lines(x=fx['days'].values, y=fx['outright'].values,
                             scales={'x': xs3, 'y': ys3}, marker='circle',
                             colors=['#9467bd'], labels=['Outright'], display_legend=True)],
            axes=[bqp.Axis(scale=xs3, label='Days'),
                  bqp.Axis(scale=ys3, orientation='vertical', tick_format='.4f')],
            fig_margin={'top': 50, 'bottom': 50, 'left': 70, 'right': 20})

        for f in (f1, f2, f3):
            f.layout.width, f.layout.height = '33%', '320px'
        return w.HBox([f1, f2, f3])

    # -- handlers ----------------------------------------------------------
    def _rebuild_rolls(self, *_):
        globals()['ACTIVE_RULE'] = self.rule.value
        pc = self.curves[self.metal.value]
        i = active_idx(pc)
        opts = [(f'{r.shfe_contract} / {r.cmx_contract}', k)
                for k, r in enumerate(pc.itertuples()) if k != i]
        self.roll.options = opts
        self.roll.value = opts[0][1] if opts else None
        self._redraw()

    def _redraw(self, *_):
        from IPython.display import display, clear_output, HTML
        metal = self.metal.value
        pc    = self.curves[metal]
        hd, i = headline(pc, metal)
        back  = self.roll.value if self.roll.value is not None else None
        tbl, meta = carry(pc, metal, i, back, self.dirn.value, self.lots.value)
        net = tbl.loc['NET CARRY']
        r = pc.iloc[i]

        with self.out_head:
            clear_output(wait=True)
            h = _BANNER.format(
                bg='#1f4e79' if r['prem_pct'] >= 0 else '#7a3b1f',
                lab='PREMIUM / DISCOUNT', val=f"{r['prem_pct']:+.2f}%",
                sub=f"{r['prem_usd_oz']:+,.3f} USD/oz &nbsp;|&nbsp; "
                    f"{r['shfe_contract']} vs {r['cmx_contract']}")
            c = _BANNER.format(
                bg='#1e6b44' if net['Annualised $'] >= 0 else '#8b1f1f',
                lab=f"NET CARRY ({meta['direction']})",
                val=f"{net['Annualised %']:+.2f}% p.a.",
                sub=f"{net['Period $']:+,.0f} USD over {meta['roll_days']}d &nbsp;|&nbsp; "
                    f"{net['Annualised $']:+,.0f} USD p.a. on "
                    f"{meta['notional_usd']:,.0f} notional")
            display(HTML(h + c))
            display(hd)

        with self.out_carry:
            clear_output(wait=True)
            print(f"{meta['front']}  ->  {meta['back']}   ({meta['roll_days']} days)")
            print(f"Premium {meta['prem_front']:+.3f}%  ->  {meta['prem_back']:+.3f}%"
                  f"    [recon err {meta['reconciliation_error']:.1e}]")
            print(f"Size: {meta['lots']:,.0f} SHFE lots = {meta['oz']:,.0f} oz "
                  f"= {meta['notional_usd']:,.0f} USD notional per leg\n")
            display(tbl[['Side', 'Period %', 'Annualised %',
                         'Period $', 'Annualised $', 'Period $/lot']]
                    .round({'Period %': 4, 'Annualised %': 3, 'Period $': 0,
                            'Annualised $': 0, 'Period $/lot': 2}))

        with self.out_size:
            clear_output(wait=True)
            display(size_trade(pc, metal, self.lots.value, i, self.dirn.value))

        with self.out_chart:
            clear_output(wait=True)
            display(self._charts(metal))

    def display(self):
        from IPython.display import display
        display(self.box)


def build(curves: Dict[str, pd.DataFrame], snap) -> Dashboard:
    return Dashboard(curves, snap)

### 3.2 — Run it

In [ ]:
Dashboard(curves, snap).display()

---
## Notes

**Where to change what.** Tickers and FX convention: section 1.1. Lot sizes, quote basis,
ceilings, OI floors, active-contract rule: sections 2.1 and 2.5. Layout and colours: section 3.1.
Edit a definition cell, re-run just that cell, then re-run section 2.9 — no need to re-pull data.

**Rerunning without Bloomberg.** `snap = load('snap_latest.pkl')` after section 1.4, then skip
straight to section 2.

**Active contract.** `ACTIVE_RULE` defaults to `BBG_ACTIVE`, the month `AUA` / `AGA` points at.

**Still not covered.**

1. CNY margin and accumulated variation margin — a separate, growing CNY cash exposure distinct
   from the notional leg, and not hedged by anything in here.
2. SHFE gold and silver settle onshore CNY while the hedge is offshore CNH, so the CNY–CNH basis
   is unhedged residual. Usually tens of pips, but it has gone to several hundred in stress — and
   stress is when a premium blows through its ceiling, so the two are likely correlated.
3. Transaction costs across three legs, SHFE margin, COMEX SPAN, FX credit lines.
4. The 13% ceiling is an input, not built up from freight, VAT, delivery and financing.
5. No historical premium series, so no z-scores or entry triggers.